# Instructions for Copilot
- This is the exploration phase for the streaming pipeline.
- Your tasks:
  1. Help load any dataset(s) from streaming_demo (CSV, JSON, Parquet, or sample JSON blobs).
2. After data is loaded:
   - Infer schema
   - Detect datatypes and anomalies
   - Recommend cleaned schema for SQL streaming
   - Suggest transformations (casting, parsing, normalization)
   - Generate reusable helper functions for cleaning
   - Produce a final "schema contract" file (clean_schema.json)
3. Prepare all output so that the next notebook (streaming_demo.ipynb) can reuse it directly.
4. Provide clear explanations in markdown.


# 1. Load and Preview Local Dataset
This notebook will help you load, profile, and clean your dataset for real-time streaming. You can use any CSV, JSON, or Parquet file (e.g., Naver Shopping or Movie reviews).


In [5]:
import pandas as pd
import os

# Use the same dataset path as in streaming_demo.ipynb
# '../../02_ml_basics/data/ratings_train.txt' (tab-separated, utf-8)
data_path = '../../02_ml_basics/data/ratings_train.txt'

# Try to load as tab-separated CSV (as in streaming_demo)
def load_data(path):
    try:
        df = pd.read_csv(path, sep='\t', encoding='utf-8')
    except Exception:
        df = pd.read_csv(path, sep=',', encoding='utf-8')
    return df

try:
    df = load_data(data_path)
    print('Loaded:', data_path)
    print('Shape:', df.shape)
    display(df.head())
except Exception as e:
    print('Error loading file:', e)
    df = None

Loaded: ../../02_ml_basics/data/ratings_train.txt
Shape: (150000, 3)


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


# 2. Infer Schema, Detect Types & Anomalies
This section will automatically infer the schema, detect datatypes, and highlight any anomalies (missing values, outliers, or unexpected types).

In [6]:
if df is not None:
    print('--- Schema Inference ---')
    print(df.dtypes)
    print('\n--- Missing Values ---')
    print(df.isnull().sum())
    print('\n--- Sample Data ---')
    display(df.sample(5))
    
    # Detect anomalies: non-numeric in numeric columns, outliers, etc.
    import numpy as np
    summary = df.describe(include='all').T
    print('\n--- Summary Statistics ---')
    display(summary)
    
    # Detect columns with mixed types
    mixed_type_cols = [col for col in df.columns if df[col].apply(type).nunique() > 1]
    if mixed_type_cols:
        print('Columns with mixed types:', mixed_type_cols)
    else:
        print('No columns with mixed types detected.')
else:
    print('No data loaded.')

--- Schema Inference ---
id           int64
document    object
label        int64
dtype: object

--- Missing Values ---
id          0
document    5
label       0
dtype: int64

--- Sample Data ---


,id,document,label
57266,5937,짜증나는 영화,0
23496,6982676,Melanie당신을 진정한 사기 캐릭터로 인명합니다.,1
64624,6812532,킬링타임도 안되는 쓰레기허접영화,0
73400,3970085,흥미로운소재일수있었지만 살려내지못한 각본과 연출 그리고 출연진,0
85854,6397968,다운받은 시간이 아까워서 끝까지 봤네요... 아 정말.... 재미없어...,0



--- Summary Statistics ---


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,150000.0,NaN,NaN,NaN,6743533.221613,2919051.392089,33.0,4766881.25,7526840.5,9249435.0,10278149.0
document,149995,146182,굿,181,NaN,NaN,NaN,NaN,NaN,NaN,NaN
label,150000.0,NaN,NaN,NaN,0.498847,0.5,0.0,0.0,0.0,1.0,1.0


Columns with mixed types: ['document']


# 3. Recommend Cleaned Schema for SQL Streaming
Based on the inferred schema, this section will recommend a cleaned schema suitable for Flink SQL or Athena, and suggest transformations (casting, normalization, etc.).

In [7]:
import json

def recommend_clean_schema(df):
    """Generate a cleaned schema contract for SQL streaming (Flink/Athena)."""
    type_map = {
        'int64': 'BIGINT',
        'float64': 'DOUBLE',
        'object': 'STRING',
        'bool': 'BOOLEAN',
        'datetime64[ns]': 'TIMESTAMP',
    }
    schema = []
    for col, dtype in df.dtypes.items():
        sql_type = type_map.get(str(dtype), 'STRING')
        schema.append({
            'name': col,
            'type': sql_type
        })
    return schema

if df is not None:
    clean_schema = recommend_clean_schema(df)
    print(json.dumps(clean_schema, indent=2))
    # Save schema contract for reuse
    with open('clean_schema.json', 'w', encoding='utf-8') as f:
        json.dump(clean_schema, f, indent=2)
else:
    print('No data loaded.')

[
  {
    "name": "id",
    "type": "BIGINT"
  },
  {
    "name": "document",
    "type": "STRING"
  },
  {
    "name": "label",
    "type": "BIGINT"
  }
]


# 4. Helper Functions for Cleaning & Transformation
Reusable functions for casting, parsing, normalization, and cleaning. Use these in your ETL or streaming jobs.

In [8]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,!?\-]', '', text)
    return text

def cast_column(df, col, dtype):
    try:
        df[col] = df[col].astype(dtype)
    except Exception as e:
        print(f'Could not cast {col} to {dtype}:', e)
    return df

def normalize_column(df, col):
    if df[col].dtype in ['float64', 'int64']:
        min_val = df[col].min()
        max_val = df[col].max()
        df[col] = (df[col] - min_val) / (max_val - min_val)
    return df

def fill_missing(df, col, value):
    df[col] = df[col].fillna(value)
    return df

# 5. Save Final Schema Contract
The cleaned schema is saved as `clean_schema.json` for use in streaming and ETL jobs.